# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook will:
- Load metadata and records.
- Review available record sets and fields via their `@id`s.
- Extract and process the data.
- Apply exploratory analysis and visualize key aspects.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the `mlcroissant` metadata structure. All record sets, fields, and columns are referenced by their `@id`.

**Note:** The Croissant schema enables referencing dataset entities via their unique `@id`, improving traceability.


In [ ]:
# List all available record sets, fields, and columns by their @id
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []

if not record_sets:
    print("No record sets directly listed in metadata. Attempting to list from the schema.")
    # Explore fields manually if not present
else:
    print('Record Sets IDs:')
    for rs in record_sets:
        print(f"- {rs['@id']}")

# Instead, use dataset.record_sets() to enumerate record set definitions
record_set_ids = []
for rs in dataset.record_sets():
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    print("  Name:", rs.get('name', 'N/A'))
    print("  Description:", rs.get('description', ''))
    fields = rs.get('field', [])
    print("  Fields:")
    for f in fields:
        print(f"    - field @id: {f['@id']}", "name:", f.get('name', 'N/A'), "datatype:", f.get('dataType', 'N/A'))
    print("  Columns:")
    columns = rs.get('column', [])
    for col in columns:
        print(f"    - column @id: {col['@id']}", "name:", col.get('name', 'N/A'), "source:", col.get('source', 'N/A'))
    print("---")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all listed record sets defined via their `@id`.


In [ ]:
# Extract data from each record set
# Obtain all record set ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}")

# For demonstration, select the first record set if available
if dataframes:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"Primary RecordSet selected for downstream analysis: {primary_record_set_id}")
    df = dataframes[primary_record_set_id]
else:
    primary_record_set_id = None
    print("No record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Select a numeric field (e.g., PHQ-9 score, GAD-7 score, PSQ score) using its `@id`.
- Filter, normalize, group as shown.


In [ ]:
# Example: EDA for primary data
import numpy as np

if primary_record_set_id and not df.empty:
    # Try to identify common numeric fields
    candidate_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['phq', 'gad', 'psq', 'score', 'total'])]
    print("Candidate numeric fields:", candidate_numeric_fields)

    if candidate_numeric_fields:
        # Select one for analysis
        numeric_field = candidate_numeric_fields[0]
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field
        candidate_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'education', 'village', 'marital'])]
        group_field = candidate_group_fields[0] if candidate_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields available for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, show comparison
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
- The data loaded from FAIR² Kilifi County Mental Health survey demonstrates structured access via Croissant schema.
- Numeric scores (PHQ-9, GAD-7, or PSQ) can be processed and compared across demographic categories.
- The `mlcroissant` library enables seamless loading and processing, referencing entities by `@id` for traceability.
- The dataset is suitable for further AI/ML analysis and public health research.